# Base vs Instruction vs Reasoning models

One mental model: **each type = previous type + one more training stage.**

```
raw text --pretrain--> BASE --SFT + RLHF--> INSTRUCT --RL on reasoning--> REASONING
```

| | Base | Instruct | Reasoning |
|---|---|---|---|
| Does | continues text | follows instructions | thinks then answers |
| Roles? | no | yes | yes |
| Prompt style | few-shot completion | direct instruction | direct (no 'think step by step') |
| Endpoint | completions | chat | chat + reasoning param |
| Cost / speed | cheap / fast | mid | expensive / slow |
| Best at | fine-tune base | general tasks | math, code, logic |

Below: same prompt through each type so you can feel the difference.

## Visual overview

**3 model types = 3 training stages. Each type = previous type + one more step.**

```
  read the whole internet   taught "listen → answer"    drilled on math/code
  (predict the next token)          + be polite                (scratch-think first)
         │                             │                            │
         ▼                             ▼                            ▼
   ┌───────────┐   SFT+RLHF   ┌───────────┐     RL      ┌─────────────┐
   │   BASE    │ ───────────▶ │  INSTRUCT │ ──────────▶ │  REASONING  │
   └───────────┘              └───────────┘             └─────────────┘
   autocomplete               assistant                 specialist
   knows a lot,               follows orders,           thinks hard,
   can't answer               answers on point          great at hard problems
```

**Same question through all three:**

```
Ask:  "What is the capital of France?"

  BASE       →  "...What is the capital of Germany?"   continues, dodges the question  ✗
  INSTRUCT   →  "It's Paris."                           answers directly                ✓
  REASONING  →  [thinks...] "Paris."                    correct, but wasteful here
```

**Trap question (needs reasoning):**

```
Ask:  "Bat + ball = $1.10. Bat costs $1.00 more than the ball. How much is the ball?"

  INSTRUCT   →  "$0.10"    ✗   fast reflex → wrong
  REASONING  →  "$0.05"    ✓   scratch-thinks → right
```

**Cost / speed (cheap+fast  ◀────▶  expensive+slow):**

```
  BASE       ▓░░░░░░░░░   cheapest, fastest
  INSTRUCT   ▓▓▓▓░░░░░░   medium
  REASONING  ▓▓▓▓▓▓▓▓▓▓   most expensive (you pay for the "thinking")
```

**When to use which:**

```
  Fine-tuning / raw autocomplete research ........ BASE
  Chatbot, summarize, translate, general Q&A ..... INSTRUCT   ← default
  Hard math, complex code, multi-step planning ... REASONING
```

## Setup

One OpenAI-compatible client covers cloud (OpenAI) and local (Ollama) — same as `openai-compatible-providers.ipynb`.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

openai = OpenAI()  # cloud: reads OPENAI_API_KEY

# Local models via Ollama (must have `ollama serve` running).
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# One prompt reused across all three model types so differences are visible.
PROMPT = "What is the capital of France?"

## 1. Base model  =>  continues text, does NOT answer

A base model only predicts the next token. It has no notion of 'you asked a question'.
It just continues whatever you gave it.

- `"The capital of France is"` -> `" Paris."`  (good continuation)
- `"What is the capital of France?"` -> often continues with MORE questions, not an answer.

To get useful output from a base model you steer it with **few-shot examples** or a
**leading prefix** it can complete.

Needs a *base* checkpoint. Cloud instruct models won't show this. Pull one locally first:
```bash
ollama pull qwen2.5:0.5b-base   # '-base' = pretrained, NOT instruct-tuned
```
Then run the cell. If you don't have it, read the expected behavior in the comments.

In [10]:
!ollama pull qwen2.5:0.5b-base

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling 9785758635cf:   0% ▕                  ▏  70 KB/397 MB                  pulling manifest 
pulling 9785758635cf:   0% ▕                  ▏ 589 KB/397 MB                  pulling manifest 
pulling 9785758635cf:   1% ▕                  ▏ 5.3 MB/397 MB                  pulling manifest 
pulling 9785758635cf:   3% ▕                  ▏  12 MB/397 MB                  pulling manifest 
pulling 9785758635cf:   4% ▕                  ▏  15 MB/397 MB                  pulling manifest 
pulling 9785758635cf:   6% ▕█                 ▏  22 MB/397 MB          

In [11]:
# Base model is autocomplete. Give it a leading prefix, not a question.
BASE_MODEL = "qwen2.5:0.5b-base"  # a base (non-instruct) checkpoint

try:
    # Note: completions endpoint (raw text in, raw text out) - no roles/messages.
    r = ollama.completions.create(
        model=BASE_MODEL,
        prompt="The capital of France is",  # prefix to CONTINUE, not a question
        max_tokens=20,
    )
    print("PREFIX  ->", repr(r.choices[0].text))

    # Same base model, but fed the raw question: watch it fail to answer.
    r2 = ollama.completions.create(model=BASE_MODEL, prompt=PROMPT, max_tokens=30)
    print("QUESTION->", repr(r2.choices[0].text))
except Exception as e:
    print("Skipped (need `ollama pull qwen2.5:0.5b-base`):", e)

PREFIX  -> 'tureng\n.preventDefaults\nennent\n.preventDefault()\n.preventDefault()\n\n\n<form action= "" method=" POST"'
QUESTION-> 'France has several capitals: Paris, Lille and Lyon. The most recent capital of France was moved from Marseille in 2018.\n\n-transitional'


## 2. Instruction model  =>  follows the instruction, answers

Base + SFT on (instruction -> answer) pairs + alignment (RLHF/DPO).
Now it understands roles (`system`/`user`/`assistant`) and actually answers.
This is what almost every app uses - suitables for interactive and creative content generation.

In [12]:
# chat endpoint + role messages - the standard instruct interface.
resp = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": PROMPT},
    ],
)
print(resp.choices[0].message.content)

The capital of France is Paris.


## 3. Reasoning model  =>  thinks first, then answers

Instruct model + RL on verifiable rewards (math/code). It generates a long internal
chain-of-thought (billed as **reasoning tokens**) before the final answer. Mostly suitable for problem solving

Rules of thumb:
- Don't write 'think step by step' - the CoT is built in.
- Control depth with `reasoning_effort` (`low`/`medium`/`high`) on OpenAI o-series.
- Slower + pricier. Overkill for 'capital of France'; shines on hard multi-step problems.

The capital question below is trivial (to compare interfaces). Swap in the HARD prompt
in the next cell to see reasoning actually earn its cost.

In [18]:
REASONING_MODEL = "o4-mini"  # OpenAI o-series; adjust to a reasoning model you have access to

try:
    resp = openai.chat.completions.create(
        model=REASONING_MODEL,
        reasoning_effort="low",           # low/medium/high - how much it 'thinks'
        messages=[{"role": "user", "content": PROMPT}],
    )
    print("ANSWER:", resp.choices[0].message.content)
    # Reasoning tokens are billed but hidden - visible in usage:
    print("usage:", resp.usage)
except Exception as e:
    print("Skipped (no reasoning-model access?):", e)

ANSWER: The capital of France is Paris.
usage: CompletionUsage(completion_tokens=25, prompt_tokens=13, total_tokens=38, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


In [19]:
# Where reasoning EARNS its cost. Try this same prompt on gpt-4o-mini (instruct)
# vs o4-mini (reasoning) and compare correctness.
HARD = (
    "A bat and a ball cost $1.10 total. The bat costs $1.00 more than the ball. "
    "How much does the ball cost? Give only the number."
)

for label, model, kwargs in [
    ("instruct ", "gpt-4o-mini", {}),
    ("reasoning", REASONING_MODEL, {"reasoning_effort": "medium"}),
]:
    try:
        r = openai.chat.completions.create(
            model=model, messages=[{"role": "user", "content": HARD}], **kwargs
        )
        print(label, "->", r.choices[0].message.content.strip())
    except Exception as e:
        print(label, "-> skipped:", e)

# Expected: ball = $0.05. Instruct models often blurt '$0.10'; reasoning gets it right.

instruct  -> 0.05
reasoning -> 0.05


## 4. Budget forcing  =>  control HOW LONG it thinks (test-time scaling)

From the **s1 paper** (Muennighoff et al., 2025). A reasoning model emits hidden thinking tokens, then a delimiter (`</think>`), then the answer. Budget forcing = you manipulate that delimiter to set a thinking budget.

- **Force stop:** hit token cap -> inject `</think>` + `Final Answer:` so it commits early.
- **Force continue:** it wants to stop -> suppress `</think>`, append `Wait` -> it keeps reasoning and often catches its own mistake. Repeat N times = more compute = higher accuracy.

Needs a model with **visible** chain-of-thought so you can edit the token stream. OpenAI o-series hides it (you only get `reasoning_effort`). Open-weight R1 shows `<think>`.
```bash
ollama pull deepseek-r1
```
Cell below drives the raw completion, stops at `</think>`, and injects `Wait` to force it to think longer. Simplified for teaching (uses R1's chat tokens directly).

> **Caveat:** on a tiny model (`deepseek-r1:1.5b`) the force-continue payoff is weak — it often re-closes `</think>` immediately, so the extra `Wait` adds little. The s1 effect (think longer -> self-correct) shows clearly on larger R1 / s1-32B. Here you're seeing the *mechanic*, not the full accuracy gain.

In [20]:
# Budget forcing (force-continue). Reuses HARD + ollama client from cells above.
R1 = "deepseek-r1:1.5b"   # local reasoning model with VISIBLE <think>...</think>
MIN_WAITS = 2         # force at least this many extra 'Wait' continuations

def think_with_budget(question, min_waits=MIN_WAITS, max_tokens=400):
    # R1 opens its chain-of-thought with <think>. Drive the raw completion and
    # STOP the moment it tries to close thinking, then nudge it to keep going.
    prompt = f"<｜User｜>{question}<｜Assistant｜><think>"
    think = ""
    for i in range(min_waits + 1):
        r = ollama.completions.create(
            model=R1, prompt=prompt + think,
            stop=["</think>"], max_tokens=max_tokens,
        )
        think += r.choices[0].text
        if i < min_waits:
            think += "\nWait"        # force-continue: suppress the stop, make it rethink
            print(f"--- injected Wait #{i + 1} ---")
    # Let it close thinking and answer.
    final = ollama.completions.create(
        model=R1, prompt=prompt + think + "</think>", max_tokens=200,
    )
    return think, final.choices[0].text

try:
    think, answer = think_with_budget(HARD)
    print("\n=== THINKING (tail, with forced Waits) ===\n", think[-600:])
    print("\n=== ANSWER ===\n", answer.strip())
except Exception as e:
    print("Skipped (need `ollama pull deepseek-r1:1.5b`):", e)

--- injected Wait #1 ---
--- injected Wait #2 ---

=== THINKING (tail, with forced Waits) ===
 
First, let's define:

Let B represent the cost of the ball in dollars.

The bat costs $1.00 more than the ball, so the bat's cost is \( B + \$1.00 \).

According to the problem, the total cost of both items is \( \$1.10 \). So we can set up the equation:

\( B + (B + \$1.00) = \$1.10 \)

Simplifying the equation:

\( 2B + \$1.00 = \$1.10 \)

Subtracting \$1.00 from both sides of the equation:

\( 2B = \$0.10 \)

Finally, to find B, we divide both sides by 2:

\( B = \$0.05 \)

Wait
Wait

=== ANSWER ===
 First, let's define the cost variables for clarity.

Let:
- \( B \) be the cost of the ball in dollars.
- The bat costs \$1.00 more than the ball, so its cost is \( B + \$1.00 \).

According to the problem, the total cost of both items is \$1.10. We can set up an equation:

\( B + (B + \$1.00) = \$1.10 \)

Next, simplify the equation by combining like terms:

\( 2B + \$1.00 = \$1.10 \)

Then,

## Takeaways

1. **Base** = autocomplete. No roles. Steer with few-shot / prefixes. Raw material for fine-tuning.
2. **Instruct** = base + SFT + alignment. Follows orders, has roles. Your default workhorse.
3. **Reasoning** = instruct + RL on verifiable rewards. Thinks before answering. Pay more for hard problems only.

### Try it
- Feed the base model a 2-shot prompt (two Q/A pairs, then a third question) and watch it finally answer.
- Run the bat-and-ball cell and compare instruct vs reasoning.
- Time each call - feel the reasoning latency.